In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/IPL.csv.xls')

In [ ]:
print('Shape:', df.shape)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
# Keep only rows where innings == 2 (the chasing team's deliveries)
# .copy() creates an independent copy so we don't accidentally modify the original
df = df[df['innings'] == 2].copy()
# dropna(subset=[...]) removes rows where the specified column is missing
# runs_target is NaN in 1st innings rows — we've already removed those,
# but this is a safety check
df = df.dropna(subset=['runs_target'])
# Remove rows where city is 'Unknown' — we can't use unknown cities
# as a feature for the model
df = df[df['city'] != 'Unknown']
# Print how many rows remain after cleaning
print('Rows remaining:', len(df))

In [ ]:
# List of current active IPL franchises (10 teams)
TEAMS = ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans',
'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians',
'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru',
'Sunrisers Hyderabad']
# .isin(TEAMS) returns True only if the value is in our TEAMS list
# We apply this to BOTH batting and bowling team columns using & (AND)
# This removes defunct teams like Kochi Tuskers, Pune Warriors etc.
df = df[df['batting_team'].isin(TEAMS) & df['bowling_team'].isin(TEAMS)]
# Verify — should show exactly 10 teams
print('Teams:', sorted(df['batting_team'].unique()))

In [ ]:
df = df[df['team_balls'] > 0]
# How many more runs does the batting team need to win?
df['runs_left'] = df['runs_target'] - df['team_runs']
# How many valid balls are remaining? (T20 = 120 balls total)
# team_balls counts only legal deliveries (not wides/no-balls)
df['balls_left'] = 120 - df['team_balls']
# How many wickets does the batting team still have?
df['wickets_remaining'] = 10 - df['team_wicket']
# Current Run Rate = runs scored so far per over
# (team_balls / 6) converts balls to overs
df['crr'] = (df['team_runs'] * 6 / df['team_balls']).round(2)
# Required Run Rate = runs still needed per over remaining
df['rrr'] = (df['runs_left'] * 6 / df['balls_left']).round(2)
# Target variable: 1 if the batting team won, 0 if they lost
# This is what the model will learn to predict
df['result'] = (df['batting_team'] == df['match_won_by']).astype(int)

In [ ]:
# Select only the columns we need — 9 features + 1 target
FEATURES = ['batting_team', 'bowling_team', 'city',
'runs_left', 'balls_left', 'wickets_remaining',
'runs_target', 'crr', 'rrr', 'result']
# Create a clean copy with only these columns
final_df = df[FEATURES].copy()
# Replace infinity values (happen when dividing by zero) with NaN
# Then drop any rows that have NaN in any column
# np.inf = positive infinity, -np.inf = negative infinity
final_df = final_df.replace([np.inf, -np.inf], np.nan).dropna()
# Confirm the final shape of our clean dataset
print('Final shape:', final_df.shape)
final_df.head()

In [ ]:
# groupby('batting_team') — splits the DataFrame into groups, one per team
# ['result'] — we look at the result column for each group
# .mean() — average of result (since result is 0 or 1, mean = win percentage)
# * 100 — convert to percentage
# .round(1) — round to 1 decimal place
# .sort_values(ascending=False) — show highest win % at the top
win_pct = (final_df.groupby('batting_team')['result']
.mean().sort_values(ascending=False) * 100).round(1)
print(win_pct)

In [ ]:
# .plot(kind='bar') creates a bar chart from the Series
win_pct.plot(kind='bar', figsize=(11, 4), color='steelblue')
# Add a title to the chart
plt.title('Win % While Chasing — By Team')
# Draw a horizontal red dashed line at 50% for reference
plt.axhline(50, color='red', linestyle='--', label='50%')
# Rotate x-axis labels so team names don't overlap
plt.xticks(rotation=40, ha='right')
# Automatically adjust spacing so nothing gets cut off
plt.tight_layout()
# Display the chart
plt.show()

In [ ]:
# .to_csv() saves the DataFrame as a CSV file
# index=False means don't write the row numbers (0,1,2...) as a column
final_df.to_csv('IPL_processed_dataset.csv', index=False)
print('Saved processed_dataset.csv')
print('Rows:', len(final_df))
print()
print('✅ Day 1 complete! Download processed_dataset.csv before closing.')

In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('/content/IPL.csv.xls')
print('Shape:', df.shape)
df.head()

In [ ]:
X = final_df.drop('result', axis=1)
y = final_df['result']
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows: ', len(X_test))

In [ ]:
CATEGORICAL = ['batting_team', 'bowling_team', 'city']
NUMERIC = ['runs_left', 'balls_left', 'wickets_remaining',
'runs_target', 'crr', 'rrr']
preprocessor = ColumnTransformer([
('cat', OneHotEncoder(
sparse_output=False,
drop='first',
handle_unknown='ignore'
), CATEGORICAL),
('num', StandardScaler(), NUMERIC)
])
pipe = Pipeline([
('step1', preprocessor),
('step2', LogisticRegression(solver='liblinear', max_iter=1000))
])
print('Pipeline built!')

In [ ]:
pipe.fit(X_train, y_train)
print('Model trained!')

In [ ]:
y_pred = pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc * 100:.2f}%')

In [ ]:

pickle.dump(pipe, open('pipe.pkl', 'wb'))
print('Saved: pipe.pkl')
loaded = pickle.load(open('pipe.pkl', 'rb'))
verify_acc = accuracy_score(y_test, loaded.predict(X_test))
print(f'Reload accuracy: {verify_acc * 100:.2f}% ✓')

In [ ]:
import ipywidgets as widgets
from IPython.display import display
ALL_CITIES = sorted(df['city'].unique().tolist())

In [ ]:
TEAMS = sorted(df['batting_team'].unique().tolist())
batting_team_widget = widgets.Dropdown(
options=TEAMS,
description='Batting Team:',
disabled=False,
continuous_update=False
)
bowling_team_widget = widgets.Dropdown(
options=TEAMS,
description='Bowling Team:',
disabled=False,
continuous_update=False
)
city_widget = widgets.Dropdown(
options=ALL_CITIES,
description='City:',
disabled=False,
continuous_update=False
)
runs_left_widget = widgets.IntText(
value=50,
description='Runs Left:',
disabled=False
)
balls_left_widget = widgets.IntText(
value=30,
description='Balls Left:',
disabled=False
)
wickets_remaining_widget = widgets.IntText(
value=7,
description='Wickets Remaining:',
disabled=False
)
runs_target_widget = widgets.IntText(
value=180,
description='Target Score:',
disabled=False
)
crr_widget = widgets.FloatText(
value=9.0,
description='Current RR:',
disabled=False
)
rrr_widget = widgets.FloatText(
value=10.0,
description='Required RR:',
disabled=False
)
predict_button = widgets.Button(description='Predict Win Probability')
output_widget = widgets.Output()
display(
batting_team_widget, bowling_team_widget, city_widget,
runs_left_widget, balls_left_widget, wickets_remaining_widget,
runs_target_widget, crr_widget, rrr_widget,
predict_button, output_widget
)

In [ ]:
def on_predict_button_clicked(b):
    with output_widget:
        output_widget.clear_output()
        try:
            input_data = pd.DataFrame({
                'batting_team': [batting_team_widget.value],
                'bowling_team': [bowling_team_widget.value],
                'city': [city_widget.value],
                'runs_left': [runs_left_widget.value],
                'balls_left': [balls_left_widget.value],
                'wickets_remaining': [wickets_remaining_widget.value],
                'runs_target': [runs_target_widget.value],
                'crr': [crr_widget.value],
                'rrr': [rrr_widget.value]
            })
            prediction = pipe.predict_proba(input_data)
            win_prob_batting_team = round(prediction[0][1] * 100, 2)
            win_prob_bowling_team = round(prediction[0][0] * 100, 2)
            print(f'{batting_team_widget.value} Win Probability: {win_prob_batting_team}%')
            print(f'{bowling_team_widget.value} Win Probability: {win_prob_bowling_team}%')
        except Exception as e:
            print(f"An error occurred: {e}")
predict_button.on_click(on_predict_button_clicked)